# Setup

In [1]:
%pip install tinysim[mujoco]
%pip install stable-baselines3[extra] gymnasium

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


# Training

In [2]:
from tinysim_mujoco.unitree_a1 import UnitreeA1WalkEnv
import numpy as np

from stable_baselines3 import PPO
from gymnasium import spaces, Env


# stable-baselines3 requires wrapping environemnts with gym.Env for training
class WalkerGymWrapper(UnitreeA1WalkEnv, Env):
    def __init__(self, reward_weights, cost_weights, headless=False):
        super().__init__(reward_weights, cost_weights, headless)
        obs_high = np.array([np.inf] * self.obs_dim * 2, dtype=np.float32)
        self.observation_space = spaces.Box(-obs_high, obs_high, dtype=np.float32)
        self.action_space = spaces.Box(
            low=self.joint_limits_low,
            high=self.joint_limits_high,
            shape=(12,),
            dtype=np.float32,
        )


reward_weights = {
    "linear_vel_tracking": 10.0,
    "angular_vel_tracking": 0.1,
    "healthy": 1.0,
    "feet_airtime": 20.0,
}
cost_weights = {
    "torque": 0.2,
    "vertical_vel": 0.0,
    "xy_angular_vel": 0.5,
    "action_rate": 0.2,
    "action_sym": 2.5,
    "joint_velocity": 0.01 * 0,
    "joint_acceleration": 2.5e-7 * 0,
    "orientation": 1.0 * 0,
    "collision": 1.0 * 0,
    "default_joint_position": 0.0,
}

env = WalkerGymWrapper(reward_weights, cost_weights, headless=True)
policy_kwargs = {"net_arch": dict(pi=[256, 128], vf=[256, 128])}

model = PPO(
    "MlpPolicy",
    env,
    learning_rate=0.001,
    n_steps=512,
    batch_size=64,
    n_epochs=3,
    gamma=0.99,
    gae_lambda=0.92,
    clip_range=0.2,
    ent_coef=0.01,
    policy_kwargs=policy_kwargs,
)

model.learn(
    total_timesteps=5,
    progress_bar=True
)


Output()

c:\Users\matth\AppData\Local\Programs\Python\Python313\Lib\site-packages\stable_baselines3\common\on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


# Simple inference

In [ ]:
import time

env = WalkerGymWrapper(reward_weights, cost_weights, headless=False)
obs, _ = env.reset()

for _ in range(100):
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = env.step(action)
    time.sleep(0.05)

env.close()